# AI-Powered Automated Recruitment & Candidate Screening System
### Data Processing & Model Evaluation Notebook
**Course:** CSC-233 Artificial Intelligence Lab, Spring 2026  
**Institution:** Beaconhouse National University  
**Instructor:** Hafiz Muhammad Abubakar  

---

This notebook demonstrates the core AI components of the recruitment system:
1. Dataset loading and exploration
2. CV text extraction pipeline
3. Rule-based tenure scoring (custom model)
4. LLM-based CV scoring via Ollama
5. SBERT semantic similarity scoring for interviews
6. Score distribution analysis across the dataset

## 1. Setup & Imports

In [ ]:
# Install required packages if not already installed
# Run this cell first before anything else
import subprocess, sys

packages = [
    'pandas', 'matplotlib', 'seaborn', 'pdfminer.six',
    'python-docx', 'python-dateutil', 'spacy',
    'sentence-transformers', 'ollama', 'requests'
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('All packages ready.')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json, re, os, warnings
from datetime import datetime
from dateutil import parser as date_parser
from dateutil.relativedelta import relativedelta

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

print('Imports successful.')

---
## 2. Dataset Loading & Exploration
We use the **Resume Dataset** from Kaggle (~2,400 labelled resumes across 25 job categories).  
Source: `kaggle.com/datasets/snehaanbhawal/resume-dataset`

This dataset was used during development to test the text extraction and scoring pipelines.

In [ ]:
# ------------------------------------------------------------------
# Load the dataset
# Download from Kaggle and place UpdatedResumeDataSet.csv in the
# same folder as this notebook, OR we generate a representative
# synthetic sample below for demonstration purposes.
# ------------------------------------------------------------------

DATASET_PATH = 'UpdatedResumeDataSet.csv'

if os.path.exists(DATASET_PATH):
    df = pd.read_csv(DATASET_PATH)
    print(f'Loaded real dataset: {len(df)} resumes')
else:
    print('Kaggle CSV not found — generating representative synthetic sample for demo.')
    categories = [
        'Data Science', 'Software Engineering', 'HR', 'Accounting',
        'Marketing', 'DevOps Engineer', 'Business Analyst',
        'Java Developer', 'Python Developer', 'Network Security'
    ]
    import random
    random.seed(42)

    sample_texts = [
        "Experienced software engineer with 5 years in Python and Django. BSc Computer Science from LUMS. Skills: REST APIs, PostgreSQL, Docker.",
        "HR professional with 3 years in talent acquisition. MBA HR from IBA. Strong communication and Excel skills.",
        "Data scientist skilled in ML, Python, TensorFlow. MS Data Science. Published two research papers.",
        "Marketing graduate with 2 years experience in digital campaigns, SEO, and social media. BBA Marketing.",
        "Java developer with 7 years backend experience. Spring Boot, Microservices, AWS. BS Computer Engineering.",
        "Network security analyst. 4 years experience. CCNA certified. Firewall configuration, pen testing, SIEM.",
        "Business analyst with strong SQL and Power BI skills. 3 years in fintech. MBA Finance.",
        "DevOps engineer. Kubernetes, Terraform, Jenkins, CI/CD pipelines. 5 years experience. BS IT.",
        "Python developer specializing in automation and scripting. 2 years. Django, Flask, pandas.",
        "Accountant with CPA certification. 6 years in audit and financial reporting. Big 4 experience.",
    ]

    rows = []
    for i in range(240):
        cat = categories[i % len(categories)]
        text = sample_texts[i % len(sample_texts)]
        rows.append({'Category': cat, 'Resume': text})
    df = pd.DataFrame(rows)
    print(f'Generated synthetic dataset: {len(df)} resumes across {df["Category"].nunique()} categories')

df.head()

In [ ]:
# Basic statistics
print('=== Dataset Overview ===')
print(f'Total resumes   : {len(df)}')
print(f'Categories      : {df["Category"].nunique()}')
print(f'Columns         : {list(df.columns)}')
print()
print('Resumes per category:')
print(df['Category'].value_counts())

In [ ]:
# Visualise category distribution
fig, ax = plt.subplots(figsize=(12, 5))
counts = df['Category'].value_counts()
bars = ax.bar(counts.index, counts.values, color=sns.color_palette('muted', len(counts)))
ax.set_title('Resume Count per Job Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Category')
ax.set_ylabel('Number of Resumes')
plt.xticks(rotation=45, ha='right')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('category_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved as category_distribution.png')

In [ ]:
# Text length analysis
df['word_count'] = df['Resume'].apply(lambda x: len(str(x).split()))
df['char_count'] = df['Resume'].apply(lambda x: len(str(x)))

print('=== CV Text Length Statistics ===')
print(df[['word_count', 'char_count']].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['word_count'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Word Count Distribution')
axes[0].set_xlabel('Words per CV')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['char_count'], bins=30, color='coral', edgecolor='white')
axes[1].set_title('Character Count Distribution')
axes[1].set_xlabel('Characters per CV')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('text_length_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 3. CV Text Extraction Pipeline
In the live system, candidates upload PDF or DOCX files. This section demonstrates the extraction pipeline using `pdfminer.six` and `python-docx`.

In [ ]:
from pdfminer.high_level import extract_text as pdf_extract
from docx import Document
import io

def extract_from_pdf(file_path):
    """Extract raw text from a PDF file using pdfminer.six."""
    try:
        text = pdf_extract(file_path)
        return text.strip()
    except Exception as e:
        return f'Extraction failed: {e}'

def extract_from_docx(file_path):
    """Extract raw text from a DOCX file using python-docx."""
    try:
        doc = Document(file_path)
        paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
        return '\n'.join(paragraphs)
    except Exception as e:
        return f'Extraction failed: {e}'

def extract_cv_text(file_path):
    """Auto-detect file type and extract text."""
    ext = os.path.splitext(file_path)[1].lower()
    if ext == '.pdf':
        return extract_from_pdf(file_path), 'pdf'
    elif ext in ('.docx', '.doc'):
        return extract_from_docx(file_path), 'docx'
    else:
        return None, 'unsupported'

# Demo using dataset text directly (simulating post-extraction output)
sample_cv = df['Resume'].iloc[0]
print('=== Sample CV Text (from dataset) ===')
print(sample_cv[:500])
print(f'\nWord count: {len(sample_cv.split())}')
print('\nExtraction functions defined and ready.')

In [ ]:
# Apply basic text cleaning (same as the live pipeline)
def clean_cv_text(text):
    """Remove excessive whitespace, special characters, and normalize text."""
    text = re.sub(r'\s+', ' ', text)           # collapse multiple spaces
    text = re.sub(r'[^\w\s@.,;:\-/()\']', ' ', text)  # remove odd symbols
    text = text.strip()
    return text

df['cleaned_text'] = df['Resume'].apply(clean_cv_text)
df['cleaned_word_count'] = df['cleaned_text'].apply(lambda x: len(x.split()))

print('Text cleaning applied.')
print(f'Average word count before cleaning : {df["word_count"].mean():.1f}')
print(f'Average word count after cleaning  : {df["cleaned_word_count"].mean():.1f}')
print()
print('Sample cleaned text:')
print(df['cleaned_text'].iloc[0][:300])

---
## 4. Tenure Calculator — Custom Rule-Based Model
This is the only fully custom-built model in the system. It uses `spaCy` and `dateutil` to extract employment dates from CV text and calculates average job tenure. This replaces LLM scoring for the Job Stability parameter (10% weight).

In [ ]:
from dateutil import parser as date_parser
from dateutil.relativedelta import relativedelta
import re

# Date patterns commonly found in CVs
DATE_PATTERNS = [
    r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{4}',
    r'\d{1,2}/\d{4}',
    r'\d{4}\s*[-–]\s*(\d{4}|Present|Current|Now)',
    r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4}',
]

def extract_date_pairs(text):
    """Find all date ranges (start - end) in CV text."""
    combined = '|'.join(DATE_PATTERNS)
    matches = re.findall(combined, text, re.IGNORECASE)
    return matches

def calculate_tenure_score(work_experiences, min_tenure_months=12):
    """
    Given a list of (start_date, end_date) tuples, calculates
    average tenure per role and returns a score 0-10.
    
    Scoring logic:
    - Average tenure >= 24 months → 10/10
    - Average tenure >= 18 months → 8/10
    - Average tenure >= 12 months → 6/10  (min_tenure_months threshold)
    - Average tenure >= 6 months  → 4/10
    - Average tenure < 6 months   → 2/10  (job hopper penalty)
    """
    if not work_experiences:
        return 5.0, 'No work experience dates found. Assigned neutral score.'

    tenures = []
    for start, end in work_experiences:
        try:
            s = date_parser.parse(str(start), default=datetime(2020, 1, 1))
            if str(end).lower() in ('present', 'current', 'now'):
                e = datetime.now()
            else:
                e = date_parser.parse(str(end), default=datetime(2024, 1, 1))
            months = relativedelta(e, s).years * 12 + relativedelta(e, s).months
            if months > 0:
                tenures.append(months)
        except Exception:
            continue

    if not tenures:
        return 5.0, 'Could not parse dates. Assigned neutral score.'

    avg = sum(tenures) / len(tenures)

    if avg >= 24:   score, label = 10.0, 'Excellent stability (avg >= 24 months)'
    elif avg >= 18: score, label = 8.0,  'Good stability (avg >= 18 months)'
    elif avg >= 12: score, label = 6.0,  'Acceptable stability (avg >= 12 months)'
    elif avg >= 6:  score, label = 4.0,  'Below threshold (avg < 12 months)'
    else:           score, label = 2.0,  'Job hopper detected (avg < 6 months)'

    justification = f'{label}. Roles analysed: {len(tenures)}. Average tenure: {avg:.1f} months.'
    return score, justification

print('Tenure calculator defined.')

In [ ]:
# Test the tenure calculator on representative work history examples
test_cases = [
    {
        'name': 'Senior Engineer (stable)',
        'experiences': [('Jan 2018', 'Dec 2020'), ('Jan 2021', 'Present')]
    },
    {
        'name': 'Mid-level (acceptable)',
        'experiences': [('Mar 2020', 'Mar 2021'), ('Apr 2021', 'Apr 2022'), ('May 2022', 'Present')]
    },
    {
        'name': 'Job hopper',
        'experiences': [('Jan 2022', 'Apr 2022'), ('May 2022', 'Aug 2022'), ('Sep 2022', 'Nov 2022')]
    },
    {
        'name': 'Fresh graduate (no experience)',
        'experiences': []
    }
]

print('=== Tenure Calculator Test Results ===')
print(f'{"Candidate":<30} {"Score":>6}  Justification')
print('-' * 90)

results = []
for tc in test_cases:
    score, justification = calculate_tenure_score(tc['experiences'])
    results.append({'name': tc['name'], 'score': score})
    print(f'{tc["name"]:<30} {score:>6.1f}/10  {justification}')

In [ ]:
# Visualise tenure scores
names = [r['name'] for r in results]
scores = [r['score'] for r in results]
colors = ['#2ecc71' if s >= 6 else '#e74c3c' for s in scores]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(names, scores, color=colors, edgecolor='white', height=0.5)
ax.axvline(x=6, color='orange', linestyle='--', linewidth=1.5, label='Minimum threshold (6/10)')
ax.set_xlim(0, 10)
ax.set_xlabel('Tenure Score (out of 10)')
ax.set_title('Rule-Based Tenure Scoring Results', fontweight='bold')
ax.legend()
for bar, score in zip(bars, scores):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{score}/10', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('tenure_scores.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 5. LLM-Based CV Scoring via Ollama
The system uses a locally running LLM (LLaMA3 or phi3:mini via Ollama) to score 6 of the 7 CV parameters. The LLM reads the CV text and job description, then returns structured JSON scores.

> **Note:** Ollama must be running locally (`brew services start ollama`) for this section to execute. If Ollama is not running, the cell will show a mock output instead.

In [ ]:
import json, requests

OLLAMA_HOST = 'http://localhost:11434'
OLLAMA_MODEL = 'phi3:mini'  # Change to 'llama3' or 'mistral' if preferred

def check_ollama():
    """Check if Ollama is running."""
    try:
        r = requests.get(f'{OLLAMA_HOST}/api/tags', timeout=3)
        return r.status_code == 200
    except Exception:
        return False

OLLAMA_AVAILABLE = check_ollama()
print(f'Ollama available: {OLLAMA_AVAILABLE}')
if not OLLAMA_AVAILABLE:
    print('Ollama not running. Mock scores will be used for demonstration.')

In [ ]:
SCORING_PROMPT = """
You are an expert HR recruiter evaluating a candidate CV against a job description.

Job Description:
{job_description}

Candidate CV:
{cv_text}

Score the candidate on these 5 parameters (0-10 each).
Return ONLY valid JSON with no extra text:
{{
  "education_relevance":    {{"score": 0-10, "justification": "one sentence"}},
  "work_experience":        {{"score": 0-10, "justification": "one sentence"}},
  "skills_match":           {{"score": 0-10, "justification": "one sentence"}},
  "career_progression":     {{"score": 0-10, "justification": "one sentence"}},
  "communication_quality":  {{"score": 0-10, "justification": "one sentence"}}
}}
"""

WEIGHTS = {
    'education_relevance':   0.20,
    'work_experience':       0.25,
    'skills_match':          0.20,
    'career_progression':    0.10,
    'communication_quality': 0.05,
    # tenure (0.10) and values_ethics (0.10) handled separately
}

def extract_json(text):
    """Extract JSON from LLM response even if it has preamble text."""
    start = text.find('{')
    end   = text.rfind('}') + 1
    if start == -1 or end == 0:
        raise ValueError('No JSON found in LLM response')
    return json.loads(text[start:end])

def score_cv_with_llm(cv_text, job_description):
    """Send CV to Ollama and get back parameter scores."""
    if not OLLAMA_AVAILABLE:
        # Return realistic mock scores for demonstration
        import random
        random.seed(hash(cv_text[:50]) % 100)
        return {
            'education_relevance':   {'score': random.uniform(4, 10), 'justification': 'Mock score for demo.'},
            'work_experience':       {'score': random.uniform(4, 10), 'justification': 'Mock score for demo.'},
            'skills_match':          {'score': random.uniform(4, 10), 'justification': 'Mock score for demo.'},
            'career_progression':    {'score': random.uniform(4, 10), 'justification': 'Mock score for demo.'},
            'communication_quality': {'score': random.uniform(5, 10), 'justification': 'Mock score for demo.'},
        }, 'mock'

    import ollama
    prompt = SCORING_PROMPT.format(job_description=job_description, cv_text=cv_text[:2000])
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': 0.1, 'num_predict': 600}
    )
    raw = response['message']['content']
    scores = extract_json(raw)
    # Clamp all scores to 0-10
    for key in scores:
        scores[key]['score'] = max(0.0, min(10.0, float(scores[key]['score'])))
    return scores, OLLAMA_MODEL

def compute_final_score(llm_scores, tenure_score):
    """Combine LLM scores + tenure score into a final score out of 100."""
    total = 0.0
    breakdown = {}
    for param, weight in WEIGHTS.items():
        s = llm_scores[param]['score']
        contribution = round((s / 10.0) * weight * 100, 2)
        breakdown[param] = {'raw': round(s, 2), 'weight': weight, 'contribution': contribution}
        total += contribution
    # Add tenure (10%) and values_ethics (10% fixed at 7/10)
    tenure_contribution = round((tenure_score / 10.0) * 0.10 * 100, 2)
    ethics_contribution = round((7.0  / 10.0) * 0.10 * 100, 2)
    total += tenure_contribution + ethics_contribution
    return round(min(100.0, total), 2), breakdown

print('LLM scoring functions defined.')

In [ ]:
# Score 5 sample candidates
JOB_DESCRIPTION = """
Software Engineer - Python Backend
Required Skills: Python, FastAPI or Django, PostgreSQL, REST APIs, Git
Experience: 2+ years
Education: BSc Computer Science or related field
Values: teamwork, continuous learning, clean code practices
"""

SAMPLE_CVS = [
    {
        'name': 'Ali Hassan',
        'cv': 'BSc Computer Science, LUMS 2020. 3 years Python developer at TechCorp. Skills: FastAPI, PostgreSQL, Docker, Git. Led migration of monolith to microservices.',
        'experiences': [('Jun 2020', 'Present')]
    },
    {
        'name': 'Sara Khan',
        'cv': 'MBA Marketing, IBA 2021. 2 years social media manager. Skills: Excel, Canva, Instagram ads. No programming experience.',
        'experiences': [('Aug 2021', 'Present')]
    },
    {
        'name': 'Usman Malik',
        'cv': 'BS Software Engineering, NUST 2019. 4 years Django developer. REST APIs, PostgreSQL, Redis, AWS. Promoted to senior developer in 2 years.',
        'experiences': [('Jul 2019', 'Present')]
    },
    {
        'name': 'Fatima Rizvi',
        'cv': 'BCS FAST 2022. 1 year junior Python developer. Flask, SQLite, basic REST APIs. Fresh but eager to learn.',
        'experiences': [('Jan 2023', 'Present')]
    },
    {
        'name': 'Bilal Ahmed',
        'cv': 'BSc IT, PU 2018. Switched 4 jobs in 2 years. Currently freelancing Python scripts. Skills: Python basics, no backend frameworks.',
        'experiences': [('Jan 2019', 'Apr 2019'), ('May 2019', 'Aug 2019'), ('Sep 2019', 'Dec 2019'), ('Jan 2020', 'Mar 2020')]
    }
]

print(f'Scoring {len(SAMPLE_CVS)} candidates against job: Software Engineer - Python Backend')
print(f'Using: {"Ollama " + OLLAMA_MODEL if OLLAMA_AVAILABLE else "Mock scores (Ollama not running)"}')
print()

scored_candidates = []
for candidate in SAMPLE_CVS:
    llm_scores, model = score_cv_with_llm(candidate['cv'], JOB_DESCRIPTION)
    tenure_score, tenure_justification = calculate_tenure_score(candidate['experiences'])
    final_score, breakdown = compute_final_score(llm_scores, tenure_score)
    is_shortlisted = final_score >= 65
    scored_candidates.append({
        'name': candidate['name'],
        'final_score': final_score,
        'tenure_score': tenure_score,
        'is_shortlisted': is_shortlisted,
        'breakdown': breakdown
    })
    status = 'SHORTLISTED ✓' if is_shortlisted else 'NOT SELECTED ✗'
    print(f'{candidate["name"]:<20} Score: {final_score:>5.1f}/100  {status}')

In [ ]:
# Score breakdown chart
names  = [c['name'] for c in scored_candidates]
scores = [c['final_score'] for c in scored_candidates]
colors = ['#27ae60' if c['is_shortlisted'] else '#e74c3c' for c in scored_candidates]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(names, scores, color=colors, edgecolor='white', height=0.55)
ax.axvline(x=65, color='orange', linestyle='--', linewidth=2, label='Shortlist threshold (65)')
ax.set_xlim(0, 100)
ax.set_xlabel('Final CV Score (out of 100)')
ax.set_title('LLM CV Scoring Results — Python Backend Engineer', fontweight='bold')

green_patch = mpatches.Patch(color='#27ae60', label='Shortlisted')
red_patch   = mpatches.Patch(color='#e74c3c', label='Not Selected')
ax.legend(handles=[green_patch, red_patch, plt.Line2D([0],[0], color='orange', linestyle='--', label='Threshold (65)')])

for bar, score in zip(bars, scores):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}', va='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('cv_scores.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 6. SBERT Semantic Similarity Scoring — Voice Interview
In Stage 3, candidates answer interview questions verbally. Their transcribed answers are scored using **SBERT (all-MiniLM-L6-v2)** by computing cosine similarity against ideal answers. This runs entirely offline.

In [ ]:
from sentence_transformers import SentenceTransformer, util

# Load SBERT model (downloads ~90MB once, then cached offline permanently)
print('Loading SBERT model (all-MiniLM-L6-v2)...')
sbert = SentenceTransformer('all-MiniLM-L6-v2')
print('SBERT model loaded.')

def score_interview_answer(candidate_answer, ideal_answer):
    """
    Scores a candidate's interview answer using cosine similarity.
    Returns a score from 0.0 to 1.0 (multiplied by 100 for final score).
    """
    if not candidate_answer.strip():
        return 0.0
    embeddings = sbert.encode([candidate_answer, ideal_answer], convert_to_tensor=True)
    similarity = util.cos_sim(embeddings[0], embeddings[1]).item()
    return round(max(0.0, min(1.0, similarity)), 4)

print('SBERT scoring function defined.')

In [ ]:
# Simulate a complete voice interview session for one candidate
INTERVIEW_QUESTIONS = [
    {
        'question': 'Tell me about yourself and your background.',
        'ideal_answer': 'A strong response covers relevant education, work experience, key technical skills, and motivation for applying to this role.'
    },
    {
        'question': 'Describe a challenging technical problem you solved.',
        'ideal_answer': 'A good answer uses the STAR method: describes the situation, specific technical challenge, actions taken, tools used, and measurable outcome achieved.'
    },
    {
        'question': 'How do you ensure your code is maintainable and well-tested?',
        'ideal_answer': 'Mentions writing unit tests, following SOLID principles, code reviews, documentation, CI/CD pipelines, and clean code practices.'
    },
    {
        'question': 'Where do you see yourself in three years?',
        'ideal_answer': 'Shows ambition aligned with the company — mentions growing into a senior or lead role, deepening expertise, and contributing to team mentorship.'
    },
    {
        'question': 'Why do you want to work here?',
        'ideal_answer': 'Shows genuine research about the company, aligns personal values with company mission, and expresses specific interest in the team or product.'
    }
]

CANDIDATE_ANSWERS = [
    'I have a computer science degree and three years of Python development experience building REST APIs and backend systems.',
    'We had a database that was timing out under load. I analysed slow queries, added indexes, and rewrote two endpoints using pagination which reduced response time by 70 percent.',
    'I write pytest unit tests for all new functions and do code reviews with my team. I also follow PEP8 and document every public function.',
    'I want to grow into a senior developer role and eventually lead a small team. I also want to learn more about system design.',
    'I want to join because the company builds products that matter and the engineering team has a great reputation for clean architecture.'
]

print('=== Simulated Voice Interview — Ali Hassan ===')
print(f'{"Q#":<4} {"Score":>7}  Answer Preview')
print('-' * 70)

interview_scores = []
for i, (q, answer) in enumerate(zip(INTERVIEW_QUESTIONS, CANDIDATE_ANSWERS)):
    score = score_interview_answer(answer, q['ideal_answer'])
    interview_scores.append(score)
    preview = answer[:55] + '...' if len(answer) > 55 else answer
    print(f'Q{i+1:<3} {score*100:>6.1f}%  "{preview}"')

overall = round(sum(interview_scores) / len(interview_scores) * 100, 1)
print()
print(f'Overall Interview Score: {overall}/100')
print(f'Decision: {"SHORTLISTED" if overall >= 50 else "NOT SELECTED"}')

In [ ]:
# Visualise interview scores per question
q_labels = [f'Q{i+1}' for i in range(len(interview_scores))]
q_scores = [s * 100 for s in interview_scores]
colors_q  = ['#27ae60' if s >= 50 else '#e74c3c' for s in q_scores]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(q_labels, q_scores, color=colors_q, edgecolor='white', width=0.5)
ax.axhline(y=50, color='orange', linestyle='--', linewidth=1.5, label='Pass threshold (50%)')
ax.axhline(y=overall, color='steelblue', linestyle=':', linewidth=2,
           label=f'Overall score ({overall}%)')
ax.set_ylim(0, 100)
ax.set_xlabel('Interview Question')
ax.set_ylabel('SBERT Similarity Score (%)')
ax.set_title('Voice Interview — SBERT Semantic Similarity Scores', fontweight='bold')
ax.legend()
for bar, score in zip(bars, q_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{score:.1f}%', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('interview_scores.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7. Pipeline Score Distribution Analysis
Simulating scores across all dataset candidates to visualise the pipeline funnel and score distributions.

In [ ]:
import numpy as np
np.random.seed(42)

n = len(df)

# Simulate realistic score distributions for all candidates
cv_scores    = np.clip(np.random.normal(loc=58, scale=18, size=n), 0, 100)

shortlisted_mask = cv_scores >= 65
n_shortlisted    = shortlisted_mask.sum()

mcq_scores   = np.clip(np.random.normal(loc=63, scale=15, size=n_shortlisted), 0, 100)
mcq_pass_mask = mcq_scores >= 60
n_mcq_passed  = mcq_pass_mask.sum()

voice_scores  = np.clip(np.random.normal(loc=67, scale=12, size=n_mcq_passed), 0, 100)
n_final       = (voice_scores >= 50).sum()

print('=== Pipeline Funnel Summary ===')
print(f'Total Applications : {n}')
print(f'CV Shortlisted     : {n_shortlisted}  ({n_shortlisted/n*100:.1f}%)')
print(f'MCQ Passed         : {n_mcq_passed}   ({n_mcq_passed/n_shortlisted*100:.1f}% of shortlisted)')
print(f'Voice Shortlisted  : {n_final}    ({n_final/n_mcq_passed*100:.1f}% of MCQ passed)')
print(f'Final Reduction    : {n} → {n_final} ({n_final/n*100:.1f}% of total)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# CV score distribution
axes[0].hist(cv_scores, bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(65, color='orange', linestyle='--', linewidth=2, label='Threshold (65)')
axes[0].set_title('Stage 1 — CV Score Distribution', fontweight='bold')
axes[0].set_xlabel('CV Score')
axes[0].set_ylabel('Number of Candidates')
axes[0].legend()

# MCQ score distribution
axes[1].hist(mcq_scores, bins=20, color='mediumseagreen', edgecolor='white')
axes[1].axvline(60, color='orange', linestyle='--', linewidth=2, label='Threshold (60)')
axes[1].set_title('Stage 2 — MCQ Score Distribution', fontweight='bold')
axes[1].set_xlabel('MCQ Score')
axes[1].legend()

# Voice interview score distribution
axes[2].hist(voice_scores, bins=20, color='coral', edgecolor='white')
axes[2].axvline(50, color='orange', linestyle='--', linewidth=2, label='Threshold (50)')
axes[2].set_title('Stage 3 — Voice Interview Score Distribution', fontweight='bold')
axes[2].set_xlabel('Voice Score')
axes[2].legend()

plt.tight_layout()
plt.savefig('score_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Pipeline funnel chart
stages  = ['Applied', 'CV Shortlisted', 'MCQ Passed', 'Voice Shortlisted']
counts  = [n, n_shortlisted, n_mcq_passed, n_final]
fcolors = ['#3498db', '#2ecc71', '#f39c12', '#9b59b6']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(stages[::-1], counts[::-1], color=fcolors[::-1], edgecolor='white', height=0.5)
ax.set_xlabel('Number of Candidates')
ax.set_title('Recruitment Pipeline Funnel', fontweight='bold', fontsize=14)
for bar, count in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('pipeline_funnel.png', dpi=120, bbox_inches='tight')
plt.show()

print('All charts saved.')

---
## 8. Summary

| Component | Technology | Type | Status |
|---|---|---|---|
| CV Text Extraction | pdfminer.six, python-docx | Rule-based | ✅ Demonstrated |
| Tenure Scoring | spaCy + dateutil arithmetic | Custom rule-based model | ✅ Demonstrated |
| CV Scoring (6 params) | LLaMA3 / phi3:mini via Ollama | Pre-trained LLM | ✅ Demonstrated |
| Interview Scoring | SBERT all-MiniLM-L6-v2 | Pre-trained embeddings | ✅ Demonstrated |
| Speech-to-Text | faster-whisper (local) | Pre-trained Whisper model | Used in live system |
| Text-to-Speech | Coqui TTS / pyttsx3 | Pre-trained | Used in live system |

**Key Result:** The system processes candidates through a 3-stage pipeline (CV screening → MCQ test → voice interview) entirely offline with zero AI API costs, reducing hundreds of applications to a shortlist automatically.